# Wine Quality Prediction System (Section B)
Run the cells **one by one, from top to bottom** (click the play button on each cell).

## Step 1: Create the project files

In [ ]:
%%writefile requirements.txt
pandas
scikit-learn
streamlit
joblib

In [ ]:
%%writefile train_model.py
# train_model.py - Wine Quality Prediction System (Section B)
# Run with: python train_model.py

import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ---------------- Activity A: Retrieve and inspect the dataset ----------------

# Download the red-wine CSV from the UCI repository (columns are separated by ";")
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep=";")

# Save a local copy (the Streamlit app reads this file)
df.to_csv("wine_quality.csv", index=False)

# Inspect the data
print("Rows and columns:", df.shape)
print(df.head())
print(df.dtypes)
print("Missing values per column:")
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Quality distribution:")
print(df["quality"].value_counts().sort_index())

# ---------------- Activity B: Preprocess, train, and evaluate ----------------

# Remove duplicate records
df = df.drop_duplicates().copy()

# Target: 1 = Good quality (quality >= 6), 0 = Needs improvement (quality < 6)
y = (df["quality"] >= 6).astype(int)

# Inputs: every column EXCEPT quality (keeping quality would be target leakage)
X = df.drop(columns=["quality"])

# Split 80% train / 20% test, keeping the same class proportions in both
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# One pipeline: fill missing values -> scale numbers -> Random Forest
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(
        n_estimators=150,
        random_state=42,
        class_weight="balanced"
    ))
])

# Train the model
model.fit(X_train, y_train)

# Test the model on data it has never seen
predictions = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
print("Confusion matrix:")
print(confusion_matrix(y_test, predictions))
print(classification_report(
    y_test, predictions,
    target_names=["Needs improvement", "Good quality"],
    zero_division=0
))

# Save the trained model for the Streamlit app
joblib.dump(model, "wine_model.pkl")
print("Saved wine_model.pkl")

In [ ]:
%%writefile app.py
# app.py - Wine Quality Prediction System (Section B)
# Run with: streamlit run app.py

import streamlit as st
import pandas as pd
import joblib

st.set_page_config(
    page_title="Wine Quality Predictor",
    page_icon="🍷",
    layout="wide"
)


@st.cache_resource
def load_model():
    # Load the trained model only once
    return joblib.load("wine_model.pkl")


model = load_model()
data = pd.read_csv("wine_quality.csv")

st.title("🍷 Wine Quality Prediction System")
st.caption("Section B classroom prototype for preliminary quality classification")

# Basic dataset information
c1, c2, c3 = st.columns(3)
c1.metric("Records", len(data))
c2.metric("Average quality score", round(data["quality"].mean(), 2))
c3.metric("Good-quality records", int((data["quality"] >= 6).sum()))

st.subheader("Dataset preview")
st.dataframe(data.head(10), use_container_width=True)

st.subheader("Predict uploaded wine samples")
st.write("Upload a CSV containing the same predictor columns as the training data.")

# Sample file the user can download and upload back
template = data.drop(columns=["quality"]).head(5)
st.download_button(
    "Download sample input CSV",
    template.to_csv(index=False),
    "sample_wine_input.csv",
    "text/csv"
)

uploaded = st.file_uploader("Choose wine sample CSV", type=["csv"])

if uploaded is not None:
    new_data = pd.read_csv(uploaded)

    # Columns the model was trained on
    required_columns = list(model.feature_names_in_)
    missing = [c for c in required_columns if c not in new_data.columns]

    if missing:
        st.error("Missing required columns: " + ", ".join(missing))
    else:
        predictions = model.predict(new_data[required_columns])

        results = new_data.copy()
        results["Prediction"] = [
            "GOOD QUALITY" if value == 1 else "NEEDS IMPROVEMENT"
            for value in predictions
        ]

        st.subheader("Prediction results")
        st.dataframe(results, use_container_width=True)
        st.metric("Predicted good-quality samples", int((predictions == 1).sum()))

        st.download_button(
            "Download prediction results",
            results.to_csv(index=False),
            "wine_quality_predictions.csv",
            "text/csv"
        )

st.info(
    "Responsible use: this is a classroom prototype. A prediction is not a "
    "substitute for laboratory testing, sensory evaluation, food-safety "
    "procedures, or professional quality-control decisions."
)

## Step 2: Install packages and train the model
This downloads the dataset, prints the evaluation results, and creates `wine_quality.csv` and `wine_model.pkl`.

In [ ]:
!pip install -q -r requirements.txt
!python train_model.py

## Step 3: Run the Streamlit app
After running this cell:
1. Copy the **IP number** it prints (this is the tunnel password).
2. Click the `https://....loca.lt` link.
3. Paste the IP number as the password and click Submit.

In [ ]:
!npm install -s localtunnel
!streamlit run app.py &>/content/logs.txt &
!sleep 5
!curl -s ipv4.icanhazip.com
!npx localtunnel --port 8501

## Step 4: Download your files (optional)
Use this to get the files for GitHub.

In [ ]:
from google.colab import files
for f in ['train_model.py','app.py','requirements.txt','wine_quality.csv','wine_model.pkl']:
    files.download(f)